In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if r.returncode != 0:
    print("❌ No GPU.\n   Runtime → Change runtime type → T4 GPU")
else:
    print(r.stdout[:500])
    print("✅ GPU ready")

In [ ]:
!pip install -q \
    "transformers>=4.52.0" \
    "timm>=1.0.9" \
    "ultralytics>=8.3.0" \
    "tqdm>=4.67.0" \
    "accelerate>=0.26.0"

import numpy as np
import torch
import cv2
from PIL import Image
from pathlib import Path
from collections import deque
import os

print(f"numpy      : {np.__version__}")
print(f"torch      : {torch.__version__}")
print(f"cv2        : {cv2.__version__}")
print(f"cuda       : {torch.cuda.is_available()}")
print("✅ All dependencies OK")

In [ ]:
CFG = {
    # ── Models ──
    "depth_model_id":         "apple/DepthPro-hf",
    "yolo_model":             "yolov8n.pt",
    "yolo_conf":              0.35,

    # ── Video ──
    "resize_width":           518,
    "output_fps":             15,

    # ── Depth colormap ──
    "depth_pct_low":          2,
    "depth_pct_high":         95,

    # ── Optical flow viz ──
    "flow_arrow_step":        30,

    # ── Gap detection ──
    "gap_min_width_px":       50,

    # ── Ego signal thresholds ──
    "fov_deg":                70.0,
    "ego_background_frac":    0.10,
    "flow_lateral_thresh":    0.65,
    "flow_rotation_thresh":   0.65,
    "flow_mag_still":         0.30,
    "flow_mag_very_slow":     0.80,
    "flow_mag_slow":          2.00,
    "flow_mag_normal":        4.50,

    # ── Agent tracking ──
    "motion_window":          8,

    # ── Risk / TTC thresholds ──
    "ttc_emergency":          1.0,
    "ttc_critical":           2.0,
    "ttc_urgent":             4.0,
    "depth_sigma_reliable":   0.10,
    "depth_sigma_moderate":   0.25,
    "gap_comfortable":        1.2,
    "gap_tight":              0.6,
    "gap_squeeze":            0.3,
    "close_distance_m":       3.0,

    # ── Zone fractions (top → bottom) ──
    "zone_overhead_top_frac": 0.18,
    "zone_overhead_bot_frac": 0.38,
    "zone_surface_top_frac":  0.62,
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── Upload your video ──────────────────────────────────────────────────────────
from google.colab import files
print("\nUpload your video file:")
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print(f"✅ Loaded: {VIDEO_PATH}")

In [ ]:
from ultralytics import YOLO
from transformers import DepthProImageProcessorFast, DepthProForDepthEstimation
from torchvision.models.optical_flow import raft_small, Raft_Small_Weights

# ── Depth Pro ─────────────────────────────────────────────────────────────────
print("Loading DepthPro...")
depth_processor = DepthProImageProcessorFast.from_pretrained(CFG["depth_model_id"])
depth_model     = DepthProForDepthEstimation.from_pretrained(CFG["depth_model_id"])
depth_model     = depth_model.to(DEVICE).eval()
print("✅ DepthPro ready")

# ── RAFT-Small Optical Flow ───────────────────────────────────────────────────
print("Loading RAFT-Small...")
_raft_weights    = Raft_Small_Weights.DEFAULT
_raft_transforms = _raft_weights.transforms()
raft_model       = raft_small(weights=_raft_weights).to(DEVICE).eval()
print("✅ RAFT-Small ready")

# ── YOLOv8 ────────────────────────────────────────────────────────────────────
print(f"Loading YOLO ({CFG['yolo_model']})...")
yolo_model    = YOLO(CFG["yolo_model"])
track_history = {}
print("✅ YOLO ready")

print("\n✅ All models loaded")

In [ ]:
import math
from collections import deque

# ── COCO class names ──────────────────────────────────────────────────────────
CLS_NAME = {
    0:"person", 1:"bicycle", 2:"car", 3:"motorcycle", 5:"bus",
    6:"train", 7:"truck", 9:"traffic_light", 10:"fire_hydrant",
    11:"stop_sign", 16:"dog", 17:"horse", 56:"chair", 57:"couch",
}
for i in range(80):
    CLS_NAME.setdefault(i, f"obj{i}")

# ── Colours ───────────────────────────────────────────────────────────────────
def _cls_to_color(c):
    if c == 0:             return (0, 220, 80)
    if c in [1, 3]:        return (255, 165, 0)
    if c == 2:             return (0, 180, 255)
    if c in [5, 6]:        return (160, 0, 255)
    if c == 7:             return (255, 0, 120)
    if c in range(14, 24): return (0, 200, 255)
    return (160, 160, 160)

CLS_COLOR = {i: _cls_to_color(i) for i in range(80)}

MOTION_COLOR = {
    "static":           (160, 160, 160),
    "approaching":      (0,   50,  230),
    "parallel_counter": (0,   0,   200),
    "receding":         (0,   200,  80),
    "parallel_same":    (0,   160,  60),
    "crossing-R":       (0,   140, 255),
    "crossing-L":       (0,   100, 200),
    "stopping":         (0,   80,  180),
    "starting":         (100, 200, 255),
    "erratic":          (0,   0,   150),
    "new":              (200, 200, 200),
}

ACTION_COLOR = {
    "walking_fast":      (0,   200,  80),
    "walking_forward":   (0,   180,  60),
    "walking_slow":      (0,   160,  50),
    "walking_very_slow": (0,   140,  40),
    "standing_still":    (120, 120, 120),
    "stopping":          (180, 180,  40),
    "stepping_left":     (255, 160,   0),
    "stepping_right":    (255, 120,   0),
    "turning_left":      (255,  60,  60),
    "turning_right":     (200,  40,  40),
}

# ══════════════════════════════════════════════════════════════════
# FLOW SIGNALS  (ego motion extraction)
# ══════════════════════════════════════════════════════════════════

def compute_flow_signals(flow_tensor, depth_arr, H, W):
    """Extract ego-motion signals from RAFT flow tensor [2,H,W]."""
    flow_np = flow_tensor.permute(1, 2, 0).cpu().numpy()   # [H,W,2]
    fx, fy  = flow_np[..., 0], flow_np[..., 1]
    mag     = np.sqrt(fx**2 + fy**2).astype(np.float32)

    bg_rows            = max(1, int(H * CFG["ego_background_frac"]))
    ego_flow_magnitude = float(mag[:bg_rows, :].mean())

    total_mag       = float(mag.sum()) + 1e-8
    xs              = np.tile(np.arange(W), (H, 1)).astype(np.float32)
    flow_centroid_x = float((mag * xs).sum() / total_mag) / W

    fx_e = float((fx**2).sum())
    fy_e = float((fy**2).sum())
    flow_lateral_ratio = fx_e / (fx_e + fy_e + 1e-8)

    left_fx  = float(fx[:, :W//2].mean())
    right_fx = float(fx[:, W//2:].mean())
    same_dir = (np.sign(left_fx) == np.sign(right_fx)) and \
               (abs(left_fx) > 0.5) and (abs(right_fx) > 0.5)
    rot_mag  = abs(left_fx - right_fx)
    max_pos  = max(abs(left_fx) + abs(right_fx), 1e-8)
    flow_rotation_score = float(1.0 - rot_mag / max_pos) if same_dir else 0.0

    return {
        "ego_flow_magnitude":  round(ego_flow_magnitude, 3),
        "flow_centroid_x":     round(flow_centroid_x, 3),
        "flow_lateral_ratio":  round(flow_lateral_ratio, 3),
        "flow_rotation_score": round(flow_rotation_score, 3),
    }

# ══════════════════════════════════════════════════════════════════
# EGO ACTION CLASSIFICATION
# ══════════════════════════════════════════════════════════════════

def _classify_camera_action(ego_mag, ego_speed, lat_ratio, centroid_x, rot_score, speed_trend):
    if ego_mag < CFG["flow_mag_still"]:
        return "standing_still", 0.95
    if ego_speed < 0.15 and speed_trend < -0.25:
        return "stopping", 0.88
    if rot_score > CFG["flow_rotation_thresh"] and ego_mag > CFG["flow_mag_still"]:
        if centroid_x > 0.52:    return "turning_left",  0.78
        elif centroid_x < 0.48:  return "turning_right", 0.78
    is_lateral = (
        lat_ratio > CFG["flow_lateral_thresh"] and
        ego_mag   > CFG["flow_mag_very_slow"]  and
        rot_score < CFG["flow_rotation_thresh"]
    )
    if is_lateral:
        if centroid_x > 0.65:    return "stepping_left",  0.82
        elif centroid_x < 0.35:  return "stepping_right", 0.82
    if ego_mag > CFG["flow_mag_normal"]:      return "walking_fast",      0.90
    elif ego_mag > CFG["flow_mag_slow"]:      return "walking_forward",   0.88
    elif ego_mag > CFG["flow_mag_very_slow"]: return "walking_slow",      0.85
    else:                                     return "walking_very_slow", 0.80

def derive_ego_signals(flow_signals, depth_arr, speed_history, H, W):
    ego_mag = flow_signals["ego_flow_magnitude"]
    cx_norm = flow_signals["flow_centroid_x"]
    lat_rat = flow_signals["flow_lateral_ratio"]
    rot_sc  = flow_signals["flow_rotation_score"]

    fov_rad   = math.radians(CFG["fov_deg"])
    focal_px  = W / (2 * math.tan(fov_rad / 2))
    med_depth = float(np.clip(np.median(depth_arr), 0.5, 12.0))
    ego_speed = round(ego_mag * med_depth / (focal_px + 1e-8) * 30.0, 3)

    speed_trend = round(ego_speed - speed_history[-1], 3) if speed_history else 0.0

    offset = cx_norm - 0.5
    if abs(offset) < 0.08:   lateral_drift = "straight"
    elif offset > 0.15:      lateral_drift = "strong_left"
    elif offset > 0.08:      lateral_drift = "slight_left"
    elif offset < -0.15:     lateral_drift = "strong_right"
    else:                    lateral_drift = "slight_right"

    action, confidence = _classify_camera_action(
        ego_mag, ego_speed, lat_rat, cx_norm, rot_sc, speed_trend)

    return {
        "ego_speed_ms":        ego_speed,
        "ego_flow_magnitude":  ego_mag,
        "flow_centroid_x":     cx_norm,
        "flow_lateral_ratio":  lat_rat,
        "flow_rotation_score": rot_sc,
        "lateral_drift":       lateral_drift,
        "speed_trend_ms":      speed_trend,
        "action":              action,
        "action_confidence":   confidence,
    }

# ══════════════════════════════════════════════════════════════════
# AGENT MOTION CLASSIFICATION  (11-class taxonomy)
# ══════════════════════════════════════════════════════════════════

def _classify_motion(tid, cx, cy, fi, depth_m):
    if tid not in track_history:
        track_history[tid] = deque(maxlen=CFG["motion_window"])
    track_history[tid].append((cx, cy, fi, depth_m))
    hist = track_history[tid]

    if len(hist) < 4:
        return "new"

    x0, y0, _, d0 = hist[0]
    xn, yn, _, dn = hist[-1]
    dx = xn - x0;  dy = yn - y0
    dist_px = (dx**2 + dy**2)**0.5

    if dist_px < 5:
        return "static"

    depth_delta  = (dn - d0) if (d0 > 0 and dn > 0) else 0
    lateral_dom  = abs(dx) > abs(dy) * 1.2
    depth_dom    = abs(depth_delta) > 0.3 and not lateral_dom

    # Erratic: frequent direction changes
    if len(hist) >= 6:
        angles = []
        for i in range(1, len(hist)):
            px2, py2 = hist[i-1][0], hist[i-1][1]
            cx2, cy2 = hist[i][0],   hist[i][1]
            if (cx2-px2)**2 + (cy2-py2)**2 > 4:
                angles.append(math.degrees(math.atan2(cy2-py2, cx2-px2)))
        if len(angles) >= 4 and float(np.std(np.diff(angles))) > 60:
            return "erratic"

    recent_spd  = ((hist[-1][0]-hist[-2][0])**2 + (hist[-1][1]-hist[-2][1])**2)**0.5
    earlier_spd = ((hist[-3][0]-hist[-4][0])**2 + (hist[-3][1]-hist[-4][1])**2)**0.5
    if recent_spd < 3 and earlier_spd > 8:  return "stopping"
    if recent_spd > 8 and earlier_spd < 3:  return "starting"

    if lateral_dom:
        return "crossing-R" if dx > 0 else "crossing-L"

    if depth_dom:
        if depth_delta < 0:
            return "parallel_counter" if abs(dx) < 15 else "approaching"
        else:
            return "parallel_same" if abs(dx) < 15 else "receding"

    if dy > 8:    return "approaching"
    elif dy < -8: return "receding"
    return "static"

def _agent_risk_level(motion, dist_m, ttc_s):
    if ttc_s is not None and ttc_s < CFG["ttc_emergency"]:  return "critical"
    if dist_m > 0 and dist_m < 1.0 and motion in ("approaching", "parallel_counter"):
        return "critical"
    if ttc_s is not None and ttc_s < CFG["ttc_critical"]:   return "high"
    if ttc_s is not None and ttc_s < CFG["ttc_urgent"]:     return "moderate"
    if motion == "erratic":                                   return "moderate"
    return "low"

# ══════════════════════════════════════════════════════════════════
# DETECT + ANNOTATE
# ══════════════════════════════════════════════════════════════════

def detect_and_annotate(frame_bgr, depth_arr, frame_idx=0):
    """
    Run YOLOv8 tracking. Returns:
      ann   — annotated BGR frame with boxes, [motion] labels, arrows, risk dots
      dets  — list of detection dicts
    """
    H, W = frame_bgr.shape[:2]
    try:
        results = yolo_model.track(
            frame_bgr, conf=CFG["yolo_conf"],
            verbose=False, persist=True, tracker="bytetrack.yaml")[0]
    except Exception:
        results = yolo_model.predict(
            frame_bgr, conf=CFG["yolo_conf"], verbose=False)[0]

    ann  = frame_bgr.copy()
    dets = []

    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls = int(box.cls[0])
        cf  = float(box.conf[0])
        cx  = (x1 + x2) // 2
        cy  = (y1 + y2) // 2
        tid = int(box.id[0]) if box.id is not None else -1

        # Depth at foot-patch (lower 40% of box)
        fy1   = y1 + int((y2 - y1) * 0.6)
        patch = depth_arr[max(fy1,0):y2, max(x1,0):x2]
        if patch.size > 0:
            dist_m       = round(float(np.percentile(patch, 10)), 2)
            dist_m_sigma = round(float(np.std(patch)), 3)
        else:
            dist_m       = round(max(0.3, min(6.0, (1.0 - y2/H) * 9.0)), 2)
            dist_m_sigma = -1.0

        motion = _classify_motion(tid, cx, cy, frame_idx, dist_m) if tid >= 0 else "static"

        # Approach speed + TTC from depth history
        hist = track_history.get(tid, [])
        if len(hist) >= 4:
            d0 = hist[0][3];  dn = hist[-1][3]
            dt = len(hist) / 30.0
            approach_speed = round((d0 - dn) / (dt + 1e-8), 3) \
                             if (d0 > 0 and dn > 0) else 0.0
        else:
            approach_speed = 0.0
        ttc = round(dist_m / (approach_speed + 1e-8), 2) \
              if approach_speed > 0.1 and dist_m > 0 else None
        if ttc: ttc = min(ttc, 99.0)

        risk = _agent_risk_level(motion, dist_m, ttc)

        # ── Draw ──────────────────────────────────────────────────────
        col       = CLS_COLOR.get(cls, (180,180,180))
        mc        = MOTION_COLOR.get(motion, (180,180,180))
        thickness = 3 if risk == "critical" else 2 if risk == "high" else 1
        cv2.rectangle(ann, (x1,y1), (x2,y2), col, thickness)

        # Label:  #id  classname  [motion]  dist
        lbl = (f"#{tid} {CLS_NAME.get(cls,'?')} [{motion}] {dist_m}m"
               if tid >= 0 else
               f"{CLS_NAME.get(cls,'?')} [{motion}] {dist_m}m")
        (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.38, 1)
        cv2.rectangle(ann, (x1, y1-th-5), (x1+tw+4, y1), col, -1)
        cv2.putText(ann, lbl, (x1+2, y1-3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0,0,0), 1)

        # Motion arrow (predicted trajectory direction)
        if len(hist) >= 4:
            x0p, y0p = hist[-4][0], hist[-4][1]
            ax = max(0, min(W-1, cx + int((cx-x0p)*3)))
            ay = max(0, min(H-1, cy + int((cy-y0p)*3)))
            if (ax, ay) != (cx, cy):
                cv2.arrowedLine(ann, (cx,cy), (ax,ay), mc, 2, tipLength=0.4)

        # Risk badge dot (top-right corner of box)
        RISK_COL = {"critical":(0,0,220), "high":(0,100,255),
                    "moderate":(0,200,255), "low":(80,180,80)}
        cv2.circle(ann, (x2-6, y1+6), 5, RISK_COL.get(risk,(120,120,120)), -1)

        dets.append({
            "cls": cls, "name": CLS_NAME.get(cls,"?"),
            "conf": round(cf,2), "box": [x1,y1,x2,y2],
            "tid": tid, "motion": motion,
            "cx": cx, "cy": cy,
            "dist_m": dist_m, "dist_m_sigma": dist_m_sigma,
            "approach_speed_ms": approach_speed,
            "ttc_s": ttc, "risk_level": risk,
            "track_age": len(hist),
        })

    return ann, dets

print("✅ Signal & detection functions ready")

In [ ]:
import math
from tqdm.notebook import tqdm
from google.colab import files as colab_files

# ══════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════

def resize_frame(frame, width):
    h, w = frame.shape[:2]
    new_w = (width // 14) * 14
    new_h = int(h * new_w / w)
    new_h = (new_h // 14) * 14
    return cv2.resize(frame, (new_w, new_h))

def estimate_depth(frame_bgr, H, W):
    pil    = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    inputs = depth_processor(images=[pil], return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = depth_model(**inputs)
    post = depth_processor.post_process_depth_estimation(outputs, target_sizes=[(H,W)])
    return post[0]["predicted_depth"].cpu().numpy().astype(np.float32)

def depth_colormap(depth_arr):
    lo  = np.percentile(depth_arr, CFG["depth_pct_low"])
    hi  = np.percentile(depth_arr, CFG["depth_pct_high"])
    clp = np.clip(depth_arr, lo, hi)
    n   = ((clp - lo) / (hi - lo + 1e-8) * 255).astype(np.uint8)
    return cv2.applyColorMap(n, cv2.COLORMAP_VIRIDIS)

def compute_flow(prev_bgr, curr_bgr):
    H, W = prev_bgr.shape[:2]
    ph   = (8 - H % 8) % 8
    pw   = (8 - W % 8) % 8
    def to_t(bgr):
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        t   = torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0).to(DEVICE)
        if ph > 0 or pw > 0:
            t = torch.nn.functional.pad(
                t.float(), (0,pw,0,ph), mode="replicate").to(torch.uint8)
        return t
    t1, t2   = to_t(prev_bgr), to_t(curr_bgr)
    t1n, t2n = _raft_transforms(t1, t2)
    with torch.no_grad():
        flow_list = raft_model(t1n.to(DEVICE), t2n.to(DEVICE))
    return flow_list[-1][0][:, :H, :W]   # [2,H,W] tensor

def flow_to_bgr_hue_arrows(flow, frame_bgr):
    """Hue-based HSV background + hue-coloured arrows."""
    flow_np = flow.permute(1,2,0).cpu().numpy()
    H, W    = flow_np.shape[:2]
    fx, fy  = flow_np[...,0], flow_np[...,1]

    angle    = (np.arctan2(fy, fx) + np.pi) / (2 * np.pi)
    mag      = np.sqrt(fx**2 + fy**2)
    mag_norm = np.clip(mag / (np.percentile(mag, 99) + 1e-6), 0, 1)

    hsv         = np.zeros((H, W, 3), dtype=np.uint8)
    hsv[..., 0] = (angle * 179).astype(np.uint8)   # hue = direction
    hsv[..., 1] = 220                               # saturation
    hsv[..., 2] = (mag_norm * 255).astype(np.uint8) # value = speed
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    bgr = cv2.addWeighted(frame_bgr, 0.25, bgr, 0.75, 0)

    step  = CFG["flow_arrow_step"]
    scale = 3.0
    for y in range(step//2, H, step):
        for x in range(step//2, W, step):
            dx, dy = float(fx[y,x]), float(fy[y,x])
            if math.sqrt(dx*dx + dy*dy) < 0.5:
                continue
            x2 = max(0, min(W-1, int(x + dx*scale)))
            y2 = max(0, min(H-1, int(y + dy*scale)))
            hue_local = int((math.atan2(dy,dx) + math.pi) / (2*math.pi) * 179)
            arrow_bgr = cv2.cvtColor(
                np.uint8([[[hue_local, 220, 230]]]),
                cv2.COLOR_HSV2BGR)[0][0].tolist()
            cv2.arrowedLine(bgr, (x,y), (x2,y2), arrow_bgr, 1, tipLength=0.35)
    return bgr

def find_gaps(depth_arr, H, W):
    surface_y = int(H * CFG["zone_surface_top_frac"])
    strip     = depth_arr[surface_y:, :]
    col_med   = np.median(strip, axis=0)
    norm      = (col_med - col_med.min()) / (col_med.max() - col_med.min() + 1e-8)
    threshold = np.percentile(norm, 55)
    open_mask = (norm >= threshold).astype(np.uint8)
    gaps = []
    in_gap, start = False, 0
    for x, val in enumerate(open_mask):
        if val and not in_gap:
            in_gap, start = True, x
        elif not val and in_gap:
            w = x - start
            if w >= CFG["gap_min_width_px"]:
                gaps.append({"cx_px": (start+x)//2, "width_px": w})
            in_gap = False
    if in_gap:
        w = W - start
        if w >= CFG["gap_min_width_px"]:
            gaps.append({"cx_px": (start+W)//2, "width_px": w})
    return gaps

def build_fusion_scene(frame_bgr, depth_arr, flow_bgr, det_frame, gaps, H, W):
    depth_col = depth_colormap(depth_arr)
    fused     = cv2.addWeighted(frame_bgr, 0.60, depth_col,  0.25, 0)
    fused     = cv2.addWeighted(fused,     0.85, flow_bgr,   0.15, 0)
    fused     = cv2.addWeighted(fused,     0.75, det_frame,  0.25, 0)

    surface_y = int(H * CFG["zone_surface_top_frac"])
    for g in gaps:
        cx, hw = g["cx_px"], g["width_px"]//2
        x1, x2 = max(0, cx-hw), min(W, cx+hw)
        overlay = fused.copy()
        overlay[surface_y:, x1:x2] = (0, 180, 0)
        fused = cv2.addWeighted(fused, 0.6, overlay, 0.4, 0)
        cv2.putText(fused, "GAP", (max(x1+4, cx-15), surface_y+20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,80), 2)

    for frac, label in [(0.18,"FREE"), (0.38,"OVERHEAD"), (0.62,"TORSO")]:
        y = int(H * frac)
        cv2.line(fused, (0,y), (W,y), (200,200,200), 1)
        cv2.putText(fused, label, (4, y-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.32, (200,200,200), 1)
    return fused

def draw_action_banner(frame, action, confidence):
    """Bottom-left pill showing ego action + confidence dot."""
    H, W  = frame.shape[:2]
    label = action.replace("_", " ")
    col   = ACTION_COLOR.get(action, (200,200,200))
    font  = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), _ = cv2.getTextSize(label, font, 0.55, 2)
    pad   = 8
    rx1, ry1 = 10,       H - th - pad*2 - 10
    rx2, ry2 = rx1+tw+pad*2, H - 10
    cv2.rectangle(frame, (rx1,ry1), (rx2,ry2), col, -1)
    cv2.rectangle(frame, (rx1,ry1), (rx2,ry2), (0,0,0), 1)
    cv2.putText(frame, label, (rx1+pad, ry2-pad), font, 0.55, (0,0,0), 2)
    conf_col = (0,200,0) if confidence>=0.85 else \
               (0,200,200) if confidence>=0.60 else (0,0,200)
    cv2.circle(frame, (rx2+10, (ry1+ry2)//2), 6, conf_col, -1)

def panel_label(img, text):
    out = img.copy()
    cv2.rectangle(out, (0,0),
                  (min(len(text)*9+10, img.shape[1]), 22), (12,12,12), -1)
    cv2.putText(out, text, (4,16),
                cv2.FONT_HERSHEY_SIMPLEX, 0.44, (220,220,220), 1)
    return out

# ══════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════

cap     = cv2.VideoCapture(VIDEO_PATH)
src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W0      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H0      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Input  : {W0}×{H0} @ {src_fps:.1f}fps  |  {total} frames")

_test = resize_frame(np.zeros((H0,W0,3), dtype=np.uint8), CFG["resize_width"])
H, W  = _test.shape[:2]
print(f"Process: {W}×{H}")

frame_skip = max(1, round(src_fps / CFG["output_fps"]))
print(f"Skip   : every {frame_skip} frames  (output ~{src_fps/frame_skip:.1f}fps)")

OUT_PATH = "output_4scene.mp4"
fourcc   = cv2.VideoWriter_fourcc(*"mp4v")
writer   = cv2.VideoWriter(OUT_PATH, fourcc, src_fps/frame_skip, (W*2, H*2))

prev_frame    = None
speed_history = []
frame_idx     = 0

with tqdm(total=total//frame_skip, desc="Processing") as pbar:
    while True:
        ret, raw = cap.read()
        if not ret:
            break
        frame_idx += 1
        if frame_idx % frame_skip != 0:
            continue

        frame = resize_frame(raw, CFG["resize_width"])

        # ── 1. Depth (needed by detection too) ────────────────────────
        depth_arr = estimate_depth(frame, H, W)
        depth_col = depth_colormap(depth_arr)

        # ── 2. Detection & Tracking  [motion labels + risk dots] ──────
        det_frame, dets = detect_and_annotate(frame, depth_arr, frame_idx)

        # ── 3. Optical Flow  (hue arrows) + Ego Action Banner ─────────
        if prev_frame is not None:
            flow_tensor  = compute_flow(prev_frame, frame)
            flow_bgr     = flow_to_bgr_hue_arrows(flow_tensor, frame)
            flow_signals = compute_flow_signals(flow_tensor, depth_arr, H, W)
            ego          = derive_ego_signals(
                               flow_signals, depth_arr, speed_history, H, W)
            speed_history = (speed_history + [ego["ego_speed_ms"]])[-3:]
        else:
            flow_bgr = np.zeros_like(frame)
            ego      = {"action": "standing_still", "action_confidence": 0.95,
                        "ego_speed_ms": 0.0}

        # Draw ego action banner onto flow panel
        flow_with_banner = flow_bgr.copy()
        draw_action_banner(flow_with_banner, ego["action"], ego["action_confidence"])

        # ── 4. Fusion Scene with Gaps ──────────────────────────────────
        gaps   = find_gaps(depth_arr, H, W)
        fusion = build_fusion_scene(frame, depth_arr, flow_bgr, det_frame, gaps, H, W)

        # ── Build 2×2 mosaic ───────────────────────────────────────────
        tl = panel_label(det_frame,        "1. Detection & Tracking  [motion] dist risk●")
        tr = panel_label(depth_col,        "2. Depth Estimation (DepthPro viridis)")
        bl = panel_label(flow_with_banner, "3. Optical Flow  hue=direction  bright=speed")
        br = panel_label(fusion,           "4. Fusion + Gaps")

        mosaic = np.vstack([np.hstack([tl, tr]),
                            np.hstack([bl, br])])
        writer.write(mosaic)
        prev_frame = frame.copy()
        pbar.update(1)

cap.release()
writer.release()
print(f"\n✅ Done → {OUT_PATH}")
colab_files.download(OUT_PATH)